# Review Volume Impact Analysis

In [1]:
import pandas as pd

# Load review volume impact output
df = pd.read_parquet("../outputs/volume_impact.parquet")
df.head()

,business_id,spike_date,window_days,spike_day_review_count,baseline_mean,abs_increase,z_score,suspicious_reviews_in_window,suspicious_user_count,total_reviews_in_impact_window,suspicious_review_share,has_suspicious_users,reviews_before_spike,reviews_on_spike_date,reviews_after_spike,after_minus_before,after_before_pct_change,volume_change_label,meaningful_increase_flag
0,VaO-VW3e1kARkU9bP1E7Fw,2019-03-09,30,7,1.966667,5.033333,4.720817,1,1,141,0.0071,1,53,7,81,28,52.83,increased,1
1,oBNrLz4EDhiscSlbOl8uAw,2019-08-18,30,10,2.633333,7.366667,5.166147,0,0,167,0.0000,0,77,10,80,3,3.90,increased,0
2,aQFQIlyHasaTz5DtjV57nQ,2019-06-02,30,10,2.233333,7.766667,4.882972,3,3,94,0.0319,1,47,10,37,-10,-21.28,decreased,0
3,FEXhWNCMkv22qG04E83Qjg,2019-12-30,30,13,2.400000,10.600000,4.106779,5,5,239,0.0209,1,56,13,170,114,203.57,increased,1
4,VaO-VW3e1kARkU9bP1E7Fw,2019-08-19,30,7,1.766667,5.233333,5.032030,2,2,82,0.0244,1,41,7,34,-7,-17.07,decreased,0


In [2]:
# Showing spike events marked as meaningful increases
df[df["meaningful_increase_flag"] == 1].head(20)

,business_id,spike_date,window_days,spike_day_review_count,baseline_mean,abs_increase,z_score,suspicious_reviews_in_window,suspicious_user_count,total_reviews_in_impact_window,suspicious_review_share,has_suspicious_users,reviews_before_spike,reviews_on_spike_date,reviews_after_spike,after_minus_before,after_before_pct_change,volume_change_label,meaningful_increase_flag
0,VaO-VW3e1kARkU9bP1E7Fw,2019-03-09,30,7,1.966667,5.033333,4.720817,1,1,141,0.0071,1,53,7,81,28,52.83,increased,1
3,FEXhWNCMkv22qG04E83Qjg,2019-12-30,30,13,2.400000,10.600000,4.106779,5,5,239,0.0209,1,56,13,170,114,203.57,increased,1
5,EtM079Cj7-B3G7jPsGYb_Q,2019-06-25,30,7,1.466667,5.533333,7.576829,0,0,57,0.0000,0,21,7,29,8,38.10,increased,1
6,kDzp5FXnuG3Pwk6orhfl8A,2019-10-22,30,9,1.133333,7.866667,22.752740,2,2,23,0.0870,1,2,9,12,10,500.00,increased,1
8,hPd6uMCjNglnOwwHpkY5Tg,2019-09-03,30,11,1.233333,9.766667,19.378040,0,0,33,0.0000,0,6,11,16,10,166.67,increased,1
10,FEXhWNCMkv22qG04E83Qjg,2019-12-27,30,8,1.666667,6.333333,3.895185,1,1,234,0.0043,1,32,8,194,162,506.25,increased,1
12,FEXhWNCMkv22qG04E83Qjg,2019-12-20,30,8,1.000000,7.000000,NaN,2,2,192,0.0104,1,6,8,178,172,2866.67,increased,1
13,oBNrLz4EDhiscSlbOl8uAw,2019-08-30,30,9,2.600000,6.400000,3.426011,0,0,171,0.0000,0,75,9,87,12,16.00,increased,1
15,FEXhWNCMkv22qG04E83Qjg,2019-12-28,30,8,1.900000,6.100000,3.067236,0,0,234,0.0000,0,39,8,187,148,379.49,increased,1
18,Ed3yyBJMOftPeqzIaQuCPA,2019-11-02,30,8,1.633333,6.366667,7.872553,0,0,47,0.0000,0,14,8,25,11,78.57,increased,1


In [3]:
df[df["meaningful_increase_flag"] == 1][[
    "reviews_before_spike",
    "reviews_on_spike_date",
    "reviews_after_spike",
    "after_minus_before",
    "after_before_pct_change",
    "suspicious_review_share",
    "suspicious_user_count"
]].describe()

,reviews_before_spike,reviews_on_spike_date,reviews_after_spike,after_minus_before,after_before_pct_change,suspicious_review_share,suspicious_user_count
count,94.000000,94.000000,94.000000,94.000000,94.000000,94.000000,94.000000
mean,31.872340,8.138298,53.585106,21.712766,138.145957,0.029735,1.414894
std,25.844545,1.418776,45.247090,31.720087,346.078010,0.049963,1.454763
min,1.000000,7.000000,4.000000,1.000000,16.000000,0.000000,0.000000
25%,8.000000,7.000000,18.000000,7.000000,33.330000,0.000000,0.000000
50%,27.000000,8.000000,45.000000,13.000000,47.725000,0.011700,1.000000
75%,49.000000,8.750000,83.500000,23.000000,112.502500,0.030300,2.000000
max,103.000000,14.000000,194.000000,172.000000,2866.670000,0.230800,7.000000


In [4]:
import pandas as pd
from pathlib import Path

# Load volume impact results
PROJECT_ROOT = Path.cwd().parent
VOLUME_PATH = PROJECT_ROOT / "outputs" / "volume_impact.parquet"

df = pd.read_parquet(VOLUME_PATH)

# Largest spike-day review events
top_spikes = df.sort_values("spike_day_review_count", ascending=False)

# Review volume increased after the spike
growth_cases = df[
    (df["volume_change_label"] == "increased") &
    (df["meaningful_increase_flag"] == 1)
].sort_values("after_before_pct_change", ascending=False).head(10)

# Spike windows with suspicious users involved
suspicious_user_cases = df[
    df["has_suspicious_users"] == 1
].sort_values("suspicious_user_count", ascending=False).head(10)

print("Meaningful Post-Spike Growth Cases")
display(growth_cases)

print("\nSpike Windows With Suspicious Users")
display(suspicious_user_cases)

Meaningful Post-Spike Growth Cases


,business_id,spike_date,window_days,spike_day_review_count,baseline_mean,abs_increase,z_score,suspicious_reviews_in_window,suspicious_user_count,total_reviews_in_impact_window,suspicious_review_share,has_suspicious_users,reviews_before_spike,reviews_on_spike_date,reviews_after_spike,after_minus_before,after_before_pct_change,volume_change_label,meaningful_increase_flag
12,FEXhWNCMkv22qG04E83Qjg,2019-12-20,30,8,1.000000,7.000000,NaN,2,2,192,0.0104,1,6,8,178,172,2866.67,increased,1
144,h-cUfr_s3U7XP2DpPkoJqg,2020-07-23,30,10,1.066667,8.933333,35.211064,0,0,29,0.0000,0,1,10,18,17,1700.00,increased,1
10,FEXhWNCMkv22qG04E83Qjg,2019-12-27,30,8,1.666667,6.333333,3.895185,1,1,234,0.0043,1,32,8,194,162,506.25,increased,1
6,kDzp5FXnuG3Pwk6orhfl8A,2019-10-22,30,9,1.133333,7.866667,22.752740,2,2,23,0.0870,1,2,9,12,10,500.00,increased,1
139,dgyKoIYEYrmRK_hktWBZdA,2020-05-03,30,7,1.233333,5.766667,13.405146,0,0,27,0.0000,0,3,7,17,14,466.67,increased,1
95,LM4-MPJIIwQlpBVZuzbJxg,2016-09-08,30,8,1.100000,6.900000,22.613418,1,1,33,0.0303,1,4,8,21,17,425.00,increased,1
15,FEXhWNCMkv22qG04E83Qjg,2019-12-28,30,8,1.900000,6.100000,3.067236,0,0,234,0.0000,0,39,8,187,148,379.49,increased,1
166,LttC5xNMFcgOg3bt_MlXTg,2018-04-06,30,8,1.000000,7.000000,NaN,3,3,13,0.2308,1,1,8,4,3,300.00,increased,1
19,FEXhWNCMkv22qG04E83Qjg,2019-12-29,30,9,2.133333,6.866667,3.024634,2,2,236,0.0085,1,47,9,180,133,282.98,increased,1
210,wj8XtPyuREj8_0GQz3LZ6w,2017-11-07,30,7,1.233333,5.766667,10.146852,2,2,50,0.0400,1,9,7,34,25,277.78,increased,1



Spike Windows With Suspicious Users


,business_id,spike_date,window_days,spike_day_review_count,baseline_mean,abs_increase,z_score,suspicious_reviews_in_window,suspicious_user_count,total_reviews_in_impact_window,suspicious_review_share,has_suspicious_users,reviews_before_spike,reviews_on_spike_date,reviews_after_spike,after_minus_before,after_before_pct_change,volume_change_label,meaningful_increase_flag
153,8V9G6dC9sLCGp1LQVrmXwA,2011-01-19,30,8,1.000000,7.000000,NaN,8,8,16,0.5000,1,4,8,4,0,0.00,no_change,0
35,XWqyoKSVyg4H8zqA8kmRBg,2021-03-15,30,7,1.433333,5.566667,7.647234,7,7,41,0.1707,1,23,7,11,-12,-52.17,decreased,0
170,pQiVbGmN4jE8QGTfrra1ew,2012-02-02,30,7,1.000000,6.000000,NaN,7,7,9,0.7778,1,0,7,2,2,NaN,increased,0
28,yttFDok9AUnK8NhzkWaEgw,2021-11-14,30,8,1.233333,6.766667,8.744109,7,7,38,0.1842,1,12,8,18,6,50.00,increased,1
70,x4p11p1v5cXJ4waelkHeFg,2018-01-23,30,7,1.533333,5.466667,5.833585,6,6,39,0.1538,1,16,7,16,0,0.00,no_change,0
202,wK2G-v74dCEzmGEUtKgOKQ,2019-06-13,30,7,1.733333,5.266667,6.709854,5,5,80,0.0625,1,33,7,40,7,21.21,increased,0
125,fq44yNPMuN0dF5ZHdDuvYw,2016-02-15,30,7,1.633333,5.366667,7.470352,5,5,37,0.1351,1,18,7,12,-6,-33.33,decreased,0
213,CGVpuQ3FpPWyYFQJQlsjLQ,2019-05-12,30,7,1.333333,5.666667,9.343558,5,5,35,0.1429,1,19,7,9,-10,-52.63,decreased,0
83,oCbJpGl8IkH1WdYa25TDBA,2018-07-07,30,7,1.600000,5.400000,7.011784,5,5,81,0.0617,1,31,7,43,12,38.71,increased,1
214,E9QzFNntGABauXJjii_Qyw,2019-08-01,30,8,1.366667,6.633333,8.202294,5,5,55,0.0909,1,18,8,29,11,61.11,increased,1


In [5]:
# Separate positive and negative review-volume impact cases
increased_cases = (
    df[df["after_minus_before"] > 0]
    .sort_values("after_minus_before", ascending=False)
)

decreased_cases = (
    df[df["after_minus_before"] < 0]
    .sort_values("after_minus_before", ascending=True)
)

print(f"Found {len(increased_cases)} spike events with increased review volume after the spike")
print(f"Found {len(decreased_cases)} spike events with decreased review volume after the spike")

print("\n--- LARGEST POST-SPIKE INCREASES ---")
display(
    increased_cases[[
        "business_id", "spike_date", "reviews_before_spike",
        "reviews_after_spike", "after_minus_before",
        "after_before_pct_change", "suspicious_user_count"
    ]].head(5)
)

print("\n--- LARGEST POST-SPIKE DECREASES ---")
display(
    decreased_cases[[
        "business_id", "spike_date", "reviews_before_spike",
        "reviews_after_spike", "after_minus_before",
        "after_before_pct_change", "suspicious_user_count"
    ]].head(5)
)

Found 136 spike events with increased review volume after the spike
Found 79 spike events with decreased review volume after the spike

--- LARGEST POST-SPIKE INCREASES ---


,business_id,spike_date,reviews_before_spike,reviews_after_spike,after_minus_before,after_before_pct_change,suspicious_user_count
12,FEXhWNCMkv22qG04E83Qjg,2019-12-20,6,178,172,2866.67,2
10,FEXhWNCMkv22qG04E83Qjg,2019-12-27,32,194,162,506.25,1
15,FEXhWNCMkv22qG04E83Qjg,2019-12-28,39,187,148,379.49,0
19,FEXhWNCMkv22qG04E83Qjg,2019-12-29,47,180,133,282.98,2
3,FEXhWNCMkv22qG04E83Qjg,2019-12-30,56,170,114,203.57,5



--- LARGEST POST-SPIKE DECREASES ---


,business_id,spike_date,reviews_before_spike,reviews_after_spike,after_minus_before,after_before_pct_change,suspicious_user_count
172,kiA1mrC2ZVoRmpsWgAT7ow,2017-03-26,54,15,-39,-72.22,0
151,_ab50qdWOk0DdB6XOrBitw,2020-03-01,73,46,-27,-36.99,0
118,GuzbBFraIq-fbkjfvaTRvg,2016-07-24,66,44,-22,-33.33,0
215,ltBBYdNzkeKdCNPDAsxwAA,2019-03-17,39,18,-21,-53.85,1
142,X0vPZIkbUj22afBQz5-neA,2015-08-01,56,37,-19,-33.93,5


In [6]:
# Suspicious businesses 
businesses_df = pd.read_parquet("../data/processed/restaurants_base.parquet")
case_studies = df.merge(businesses_df, on="business_id", how="left")
case_studies.sort_values("after_minus_before", ascending=False).head(10)

,business_id,spike_date,window_days,spike_day_review_count,baseline_mean,abs_increase,z_score,suspicious_reviews_in_window,suspicious_user_count,total_reviews_in_impact_window,...,city,hours,is_open,latitude,longitude,name,postal_code,review_count,stars,state
12,FEXhWNCMkv22qG04E83Qjg,2019-12-20,30,8,1.000000,7.000000,NaN,2,2,192,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
10,FEXhWNCMkv22qG04E83Qjg,2019-12-27,30,8,1.666667,6.333333,3.895185,1,1,234,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
15,FEXhWNCMkv22qG04E83Qjg,2019-12-28,30,8,1.900000,6.100000,3.067236,0,0,234,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
19,FEXhWNCMkv22qG04E83Qjg,2019-12-29,30,9,2.133333,6.866667,3.024634,2,2,236,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
3,FEXhWNCMkv22qG04E83Qjg,2019-12-30,30,13,2.400000,10.600000,4.106779,5,5,239,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
21,FEXhWNCMkv22qG04E83Qjg,2020-01-04,30,12,3.433333,8.566667,2.589585,1,1,258,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
108,GuzbBFraIq-fbkjfvaTRvg,2016-11-06,30,8,2.133333,5.866667,4.242036,0,0,140,...,Santa Barbara,"{'Friday': '11:0-21:0', 'Monday': None, 'Satur...",1,34.401348,-119.723080,Mesa Verde,93109,1796,4.5,CA
171,ac1AeYqs8Z4_e2X5M3if2A,2017-05-23,30,8,2.066667,5.933333,5.060492,2,2,162,...,New Orleans,"{'Friday': '8:0-1:0', 'Monday': '8:0-1:0', 'Sa...",1,29.956231,-90.067563,Oceana Grill,70130,7400,4.0,LA
27,oBNrLz4EDhiscSlbOl8uAw,2021-02-14,30,10,1.700000,8.300000,8.119995,1,1,105,...,New Orleans,"{'Friday': '7:30-14:0', 'Monday': '0:0-0:0', '...",1,29.951025,-90.067394,Ruby Slipper - New Orleans,70130,5193,4.5,LA
188,iksVwRfpWymIUUFqw0tXpw,2019-11-01,30,7,1.966667,5.033333,4.234863,0,0,116,...,Philadelphia,"{'Friday': '11:30-23:0', 'Monday': '0:0-0:0', ...",1,39.955010,-75.156134,Chubby Cattle,19107,1128,4.5,PA


In [7]:
# Counts for presentation/report summary
volume_change_counts = df["volume_change_label"].value_counts().reset_index()
volume_change_counts.columns = ["volume_change_label", "count"]

meaningful_counts = df["meaningful_increase_flag"].value_counts().reset_index()
meaningful_counts.columns = ["meaningful_increase_flag", "count"]

suspicious_counts = df["has_suspicious_users"].value_counts().reset_index()
suspicious_counts.columns = ["has_suspicious_users", "count"]

print("Volume Change Counts")
display(volume_change_counts)

print("\nMeaningful Increase Counts")
display(meaningful_counts)

print("\nSuspicious User Involvement Counts")
display(suspicious_counts)

Volume Change Counts


,volume_change_label,count
0,increased,136
1,decreased,79
2,no_change,8



Meaningful Increase Counts


,meaningful_increase_flag,count
0,0,129
1,1,94



Suspicious User Involvement Counts


,has_suspicious_users,count
0,1,141
1,0,82


In [8]:
# Top 5 spike events ranked by post-spike review increase
df.sort_values("after_minus_before", ascending=False)[[
    "business_id",
    "spike_date",
    "spike_day_review_count",
    "reviews_before_spike",
    "reviews_after_spike",
    "after_minus_before",
    "after_before_pct_change",
    "suspicious_user_count",
    "suspicious_review_share"
]].head(5)

,business_id,spike_date,spike_day_review_count,reviews_before_spike,reviews_after_spike,after_minus_before,after_before_pct_change,suspicious_user_count,suspicious_review_share
12,FEXhWNCMkv22qG04E83Qjg,2019-12-20,8,6,178,172,2866.67,2,0.0104
10,FEXhWNCMkv22qG04E83Qjg,2019-12-27,8,32,194,162,506.25,1,0.0043
15,FEXhWNCMkv22qG04E83Qjg,2019-12-28,8,39,187,148,379.49,0,0.0000
19,FEXhWNCMkv22qG04E83Qjg,2019-12-29,9,47,180,133,282.98,2,0.0085
3,FEXhWNCMkv22qG04E83Qjg,2019-12-30,13,56,170,114,203.57,5,0.0209


In [9]:
# Looking for strongest suspicious business-impact cases
# Finds spike events that had suspicious users AND meaningful post-spike review growth
strong_cases = case_studies[
    (case_studies["has_suspicious_users"] == 1) &
    (case_studies["meaningful_increase_flag"] == 1)
].sort_values(["suspicious_user_count", "after_minus_before"], ascending=False)

strong_cases.head(10)

,business_id,spike_date,window_days,spike_day_review_count,baseline_mean,abs_increase,z_score,suspicious_reviews_in_window,suspicious_user_count,total_reviews_in_impact_window,...,city,hours,is_open,latitude,longitude,name,postal_code,review_count,stars,state
28,yttFDok9AUnK8NhzkWaEgw,2021-11-14,30,8,1.233333,6.766667,8.744109,7,7,38,...,Reno,"{'Friday': '9:30-21:0', 'Monday': '0:0-0:0', '...",1,39.528762,-119.880227,Inclined Burgers And Brews,89523,80,4.5,NV
3,FEXhWNCMkv22qG04E83Qjg,2019-12-30,30,13,2.400000,10.600000,4.106779,5,5,239,...,New Orleans,"{'Friday': '8:0-20:0', 'Monday': '8:0-20:0', '...",1,29.957525,-90.061861,Café Du Monde,70116,1880,4.0,LA
83,oCbJpGl8IkH1WdYa25TDBA,2018-07-07,30,7,1.600000,5.400000,7.011784,5,5,81,...,New Orleans,"{'Friday': '11:0-23:0', 'Monday': '0:0-0:0', '...",1,29.968748,-90.051093,Morrow's,70117,831,4.0,LA
214,E9QzFNntGABauXJjii_Qyw,2019-08-01,30,8,1.366667,6.633333,8.202294,5,5,55,...,Reno,"{'Friday': '15:0-22:0', 'Monday': '0:0-0:0', '...",0,39.445495,-119.745703,Zeppelin,89521,299,4.0,NV
182,_ab50qdWOk0DdB6XOrBitw,2018-12-01,30,8,2.466667,5.533333,4.519680,4,4,153,...,New Orleans,"{'Friday': '11:0-22:0', 'Monday': '11:0-22:0',...",1,29.954273,-90.068965,Acme Oyster House,70130,7568,4.0,LA
82,9n-1LQLX3ntBfBtMwgSpig,2018-02-22,30,7,1.200000,5.800000,7.620305,4,4,23,...,Wesley Chapel,"{'Friday': '11:0-17:0', 'Monday': '0:0-0:0', '...",1,28.187716,-82.349963,900 Degrees Woodfired Pizza,33543,258,4.0,FL
154,OMT709IPPEwN91CAXpe9dw,2012-01-22,30,8,1.233333,6.766667,13.425741,4,4,30,...,Tucson,"{'Friday': '16:0-22:0', 'Monday': '16:0-21:0',...",0,32.220624,-110.968407,Downtown Kitchen + Cocktails,85701,571,4.0,AZ
221,M3yOPL5TRCU0G5o8Ciicbw,2022-01-10,30,8,1.233333,6.766667,11.906421,4,4,18,...,Nashville,"{'Friday': '12:0-3:0', 'Monday': '16:0-2:0', '...",1,36.149388,-86.797186,Rebar,37203,139,3.0,TN
96,J8S7cPPlTgsQnXKVfTyN8g,2014-09-09,30,10,1.300000,8.700000,14.598211,3,3,65,...,Philadelphia,"{'Friday': '11:0-2:0', 'Monday': '17:0-2:0', '...",0,39.966505,-75.139149,PYT,19123,806,3.0,PA
55,GXFMD0Z4jEVZBCsbPf4CTQ,2018-10-04,30,9,2.500000,6.500000,5.433608,3,3,164,...,Nashville,"{'Friday': '11:0-0:0', 'Monday': '0:0-0:0', 'S...",1,36.151387,-86.796603,Hattie B’s Hot Chicken - Nashville,37203,6093,4.5,TN


In [10]:
strong_cases[[
    "business_id",
    "name",
    "spike_date",
    "reviews_before_spike",
    "reviews_on_spike_date",
    "reviews_after_spike",
    "after_minus_before",
    "after_before_pct_change",
    "suspicious_reviews_in_window",
    "suspicious_user_count",
    "suspicious_review_share"
]].head(10)

,business_id,name,spike_date,reviews_before_spike,reviews_on_spike_date,reviews_after_spike,after_minus_before,after_before_pct_change,suspicious_reviews_in_window,suspicious_user_count,suspicious_review_share
28,yttFDok9AUnK8NhzkWaEgw,Inclined Burgers And Brews,2021-11-14,12,8,18,6,50.00,7,7,0.1842
3,FEXhWNCMkv22qG04E83Qjg,Café Du Monde,2019-12-30,56,13,170,114,203.57,5,5,0.0209
83,oCbJpGl8IkH1WdYa25TDBA,Morrow's,2018-07-07,31,7,43,12,38.71,5,5,0.0617
214,E9QzFNntGABauXJjii_Qyw,Zeppelin,2019-08-01,18,8,29,11,61.11,5,5,0.0909
182,_ab50qdWOk0DdB6XOrBitw,Acme Oyster House,2018-12-01,58,8,87,29,50.00,4,4,0.0261
82,9n-1LQLX3ntBfBtMwgSpig,900 Degrees Woodfired Pizza,2018-02-22,5,7,11,6,120.00,4,4,0.1739
154,OMT709IPPEwN91CAXpe9dw,Downtown Kitchen + Cocktails,2012-01-22,8,8,14,6,75.00,4,4,0.1333
221,M3yOPL5TRCU0G5o8Ciicbw,Rebar,2022-01-10,3,8,7,4,133.33,4,4,0.2222
96,J8S7cPPlTgsQnXKVfTyN8g,PYT,2014-09-09,12,10,43,31,258.33,3,3,0.0462
55,GXFMD0Z4jEVZBCsbPf4CTQ,Hattie B’s Hot Chicken - Nashville,2018-10-04,66,9,89,23,34.85,3,3,0.0183


In [11]:
total_events = len(df)
increased = (df["volume_change_label"] == "increased").sum()
decreased = (df["volume_change_label"] == "decreased").sum()
meaningful = df["meaningful_increase_flag"].sum()
with_suspicious_users = df["has_suspicious_users"].sum()
mean_before = df["reviews_before_spike"].mean()
mean_after = df["reviews_after_spike"].mean()
median_pct_change = df["after_before_pct_change"].median()

print(f"Total suspicious spike events analyzed: {total_events}")
print(f"Events with increased review volume after the spike: {increased}")
print(f"Events with decreased review volume after the spike: {decreased}")
print(f"Events with meaningful post-spike increase: {meaningful}")
print(f"Events involving suspicious users: {with_suspicious_users}")
print(f"Average reviews before spike: {mean_before:.2f}")
print(f"Average reviews after spike: {mean_after:.2f}")
print(f"Median percent change after spike: {median_pct_change:.2f}%")

Total suspicious spike events analyzed: 223
Events with increased review volume after the spike: 136
Events with decreased review volume after the spike: 79
Events with meaningful post-spike increase: 94
Events involving suspicious users: 141
Average reviews before spike: 35.60
Average reviews after spike: 42.70
Median percent change after spike: 12.50%
